In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/India_Overview_5_Pages_for_RAG.pdf")
docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap= 200)
splitted_docs = splitter.split_documents(docs)

In [5]:
embiddings = GoogleGenerativeAIEmbeddings(model= "gemini-embedding-2-preview")


In [7]:
vector_store = InMemoryVectorStore.from_documents(
    documents= splitted_docs,
    embedding= embiddings
)

### Now We are building the Agent,
For Agent we need --tools, llm, systempromt

In [34]:
@tool
def retriver_tool(query:str):
    """
        This Tool can help to retrive the relevant data from the PDF document
    """
    print("tool called:", query)
    docs = vector_store.similarity_search(query=query, k=4)

    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [35]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [36]:
System_Prompt = """
    You are an helpful agent that answer questions using retrived context,
    Always use the retriver_tool for questions requiring the external knowledge.

"""

In [ ]:
agent = create_agent(
    model = llm,
    tools= [retriver_tool],
    system_prompt= System_Prompt
)

In [ ]:
query = "Tell me about Economy, Technology and Modern India and also about Culture, Food and Tourism and tell me the capital of India?"
result = agent.invoke({"messages":[{"role": "user", "content": query}]})
print(result["messages"][-1].content[0]["text"])

tool called: Economy, Technology, Modern India, Culture, Food, Tourism and capital of India
tool called: Economy and technology in modern India
tool called: What is the capital of India?
tool called: Is New Delhi the capital of India?
tool called: What is the capital of India?
India is a nation that blends a deep historical heritage with a rapidly evolving modern identity. Here is an overview based on the information provided:

### **Economy, Technology, and Modern India**
Modern India is characterized by a rapidly changing economy. The country has focused on expanding its infrastructure, with significant developments in highways and aviation that connect major economic centers, alongside the growth of metro systems in large urban areas. India possesses a strong technology sector and operates under a parliamentary system of government, with a federal structure that divides powers between the Union and state governments.

### **Culture, Food, and Tourism**
*   **Culture:** Indian cultur

# Here You can see how many times the agent called the tool to answer the question that is why we use the Agentic Rag

tool called: Economy, Technology, Modern India, Culture, Food, Tourism and capital of India
tool called: Economy and technology in modern India
tool called: What is the capital of India?
tool called: Is New Delhi the capital of India?
tool called: What is the capital of India?
India is a nation that blends a deep historical heritage with a rapidly evolving modern identity. Here is an overview based on the information provided:

### **Economy, Technology, and Modern India**
Modern India is characterized by a rapidly changing economy. The country has focused on expanding its infrastructure, with significant developments in highways and aviation that connect major economic centers, alongside the growth of metro systems in large urban areas. India possesses a strong technology sector and operates under a parliamentary system of government, with a federal structure that divides powers between the Union and state governments.

### **Culture, Food, and Tourism**
*   **Culture:** Indian culture is a rich tapestry formed by interactions among many communities and traditions. It encompasses diverse forms of music (such as Hindustani and Carnatic), dance (including Bharatanatyam, Kathak, and others), literature in numerous languages, cinema, and religious traditions. Festivals—such as Diwali, Holi, Eid, and many regional celebrations—are central to social life.
*   **Food:** Indian cuisine is highly regional and diverse. Staples include rice, wheat, lentils, vegetables, dairy, and spices. Popular regional examples range from *dosa* and *idli* in South India and *dhokla* in Gujarat to *rogan josh* in Kashmir and *litti chokha* in Bihar.
*   **Tourism:** India attracts visitors with a wide variety of landmarks and landscapes. Key destinations include the Taj Mahal in Agra, historic monuments in Delhi (such as the Red Fort and Qutub Minar), the palaces and forts of Rajasthan, and the scenic landscapes of Kerala.

### **Capital of India**
The capital of India is **New Delhi** (often referred to simply as Delhi, which contains major landmarks and serves as the seat of government).